In [1]:
import numpy as np
import sys
sys.path.append('..')
import os
import torch
from torch.utils.data import TensorDataset, DataLoader
import warnings
import pickle
warnings.filterwarnings("ignore")
from collections import OrderedDict
import xarray as xr

from src.data_assemble.assemble_ml import *
from src.data_assemble.assemble_conv import *
from src.models.utils import *

/home/a.galliamov/miniconda3/envs/pygdall/lib/python3.5/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rectangle_coords = {'lat_min': 41.12, 'lat_max': 81.49,'lon_min': 19.38, 'lon_max': 169.40}
target_res = {'lon_res': 0.25, 'lat_res': 0.25}
contains = {"years": ['2016', '2026', '2036'], "bands": ['max']}
rus = dp.get_xarrays('../data/stash/WindProject/cmip_stash/', rectangle_coords, target_res, contains)
rus['max'].shape

100%|██████████| 3650/3650 [00:03<00:00, 1206.23it/s]


(10950, 168, 608)

In [3]:
rectangle_coords = {'lat_min': 43.5, 'lat_max': 47,'lon_min': 36.5, 'lon_max': 41.75}
target_res = {'lon_res': 0.25, 'lat_res': 0.25}
contains = {"years": ['2016'], "bands": ['max']}
kk = dp.get_xarrays('../data/stash/WindProject/cmip_stash/', rectangle_coords, target_res, contains)
kk['max'].shape

100%|██████████| 3650/3650 [00:00<00:00, 16493.88it/s]


In [6]:
# # ['elevation', 'pr', 'tasmax', 'tasmin', 'wind']
path_to_files = ['../data/elev/*.tif', '../data/stash/WindProject/cmip_stash/*.nc']
filter_dict = {"years": ['2006'], "bands": ['max']}
rectangle_coords = {'lat_min': 41.12, 'lat_max': 81.49,'lon_min': 19.38, 'lon_max': 169.40}
target_res = {'lon_res': 0.25, 'lat_res': 0.25}

blocks = make_blocks(path_to_files, filter_dict, rectangle_coords, target_res, half_side_size=3)

100%|██████████| 2/2 [00:44<00:00, 22.17s/it]


In [7]:
path_to_files = ['../data/elev/*.tif', '../data/stash/WindProject/cmip_stash/*.nc']
filter_dict = {"years": ['2006'], "bands": ['max']}
rectangle_coords = {'lat_min': 43.5, 'lat_max': 47,'lon_min': 36.5, 'lon_max': 41.75}
target_res = {'lon_res': 0.25, 'lat_res': 0.25}

blockskk = make_blocks(path_to_files, filter_dict, rectangle_coords, target_res, half_side_size=3)

100%|██████████| 2/2 [00:00<00:00, 20.89it/s]


In [3]:
start = '2006-01-01'
end = '2016-01-01'
df = pd.read_csv('../data_meteo_kk.csv')
station_list = pd.read_csv('../weatherstation_list.csv')

In [4]:
station_names = ['Анапа', 'Армавир', 'Краснодар, Круглик', 'Сочи', 'Туапсе', 'Приморско-Ахтарск', 'Красная Поляна']
stations_pixs = get_pixel_stations(path_to_tifs[0], feature_names[0], station_names, station_list)

In [5]:
target = get_y(df, start, end, speed_th=20)

In [6]:
X = assemble_numpy_ds(blocks=blocks, target=target, stations_pixs=stations_pixs, include_target=False)

In [7]:
# WRITE PATH TO TERRABYTE

path_to_dump = os.path.join('..', 'data','nn_data_grid_inference')
obj_path = os.path.join(path_to_dump, 'objects')
for k in X.keys():
    X_station = X[k]

    st_path = os.path.join(path_to_dump, str(k))
    if not os.path.isdir(st_path):
        os.makedirs(st_path)
# WRITE PATH TO TERRABYTE    
    with open(os.path.join(st_path, 'objects.npy'),'wb') as f:
        # pickle.dump(X_station, f)
        np.save(f, X_station)